# Talhões candidatos em Jussara-GO (Google Earth Engine + geemap)

Este notebook mapeia **talhões candidatos** no município de **Jussara-GO** com base em NDVI máximo anual (2024 e 2025), sem uso de CAR.

Pipeline:
1. Definir AOI = limite municipal.
2. Coletar Sentinel-2 SR 2024–2025 com máscara de nuvem.
3. Criar NDVI e composições mensais.
4. Gerar NDVI máximo anual (2024 e 2025).
5. Criar máscara agrícola com threshold NDVI_max > X (parametrizável) e filtros de ruído.
6. Converter raster para vetores (reduceToVectors), calcular área (ha) e filtrar por área mínima.
7. Exportar GeoJSON dos talhões para Google Drive.
8. (Opcional) Estratégia por tiles (grade) para não exceder limites.


In [ ]:
# @title 0) Instalar e importar dependências
!pip -q install geemap

import ee
import geemap

print('✅ Dependências importadas')


In [ ]:
# @title 1) Parâmetros (edite aqui)
# -------------------------------------------------------------------
# Não invente paths: defina variáveis no topo
# -------------------------------------------------------------------
import os

MUNICIPIO = 'Jussara'  # Nome do município
UF = 'GO'              # Estado
ANO_INI = 2024
ANO_FIM = 2025

# Projeto do Google Earth Engine (recomendado definir aqui)
# Exemplo: 'meu-projeto-gcp'
EE_PROJECT_ID = None

# NDVI threshold (parametrizável)
NDVI_MAX_THRESHOLD = 0.55  # ajuste conforme necessário

# Filtros de ruído (hectares)
AREA_MIN_HA = 5            # área mínima do talhão
AREA_MAX_HA = 1000         # opcional (use None para não filtrar)

# Máscara de nuvem Sentinel-2 SR
CLOUD_PROB_MAX = 60        # s2cloudless probability
S2_SCALE = 10              # resolução (m)

# Exportação
DRIVE_FOLDER = 'GEE_Export'  # pasta do Google Drive
EXPORT_PREFIX = 'talhoes_jussara_ndvi_max'

# Descoberta de projeto por variável de ambiente
if not EE_PROJECT_ID:
    EE_PROJECT_ID = os.getenv('EE_PROJECT_ID') or os.getenv('GOOGLE_CLOUD_PROJECT')

# Fallback interativo em notebook (evita erro "no project found")
if not EE_PROJECT_ID:
    EE_PROJECT_ID = input('Informe o GCP Project ID para Earth Engine: ').strip()

if not EE_PROJECT_ID:
    raise ValueError('É necessário informar um GCP Project ID para inicializar o Earth Engine.')

# Inicialização do Earth Engine (sempre com projeto explícito)
try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)

print(f'✅ Parâmetros configurados e Earth Engine inicializado (project={EE_PROJECT_ID})')


In [ ]:
# @title 2) Definir AOI (limite municipal)
# Fonte: FAO GAUL simplificado (nivel 2). Substitua se necessário.

gaul = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2')

# Filtro por município e UF
municipio = gaul.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM2_NAME', MUNICIPIO),
        ee.Filter.eq('ADM1_NAME', 'Goias')
    )
)

if municipio.size().getInfo() == 0:
    raise ValueError('Município não encontrado. Verifique MUNICIPIO/UF ou use outra fonte de limites.')

AOI = municipio.geometry()
print('✅ AOI definido:', MUNICIPIO, UF)


In [ ]:
# @title 3) Funções auxiliares: máscara de nuvem e NDVI
# Sentinel-2 SR + s2cloudless
S2_SR = 'COPERNICUS/S2_SR_HARMONIZED'
S2_CLOUDS = 'COPERNICUS/S2_CLOUD_PROBABILITY'

# Filtro de data
start_date = f'{ANO_INI}-01-01'
end_date = f'{ANO_FIM}-12-31'

# Coleções
s2_sr_col = (ee.ImageCollection(S2_SR)
             .filterDate(start_date, end_date)
             .filterBounds(AOI)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)))

s2_cloud_col = (ee.ImageCollection(S2_CLOUDS)
                .filterDate(start_date, end_date)
                .filterBounds(AOI))

# Junta SR com cloud probability
joined = ee.ImageCollection(ee.Join.saveFirst('cloud_mask').apply(
    primary=s2_sr_col,
    secondary=s2_cloud_col,
    condition=ee.Filter.equals(leftField='system:index', rightField='system:index')
))


def mask_s2_clouds(img):
    cloud_prob = ee.Image(img.get('cloud_mask')).select('probability')
    is_clear = cloud_prob.lt(CLOUD_PROB_MAX)
    # Máscara adicional para sombras (SCL)
    scl = img.select('SCL')
    mask = (scl.neq(3)  # sombra
            .And(scl.neq(7))  # água
            .And(scl.neq(8))  # nuvem média
            .And(scl.neq(9))  # nuvem alta
            .And(scl.neq(10))  # cirrus
            .And(scl.neq(11)))  # neve
    return img.updateMask(is_clear).updateMask(mask)


def add_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

print('✅ Funções auxiliares prontas')


In [ ]:
# @title 4) Coleção Sentinel-2 com NDVI
s2_ndvi = (joined
           .map(mask_s2_clouds)
           .map(add_ndvi)
           .select(['NDVI']))

print('✅ Coleção Sentinel-2 com NDVI pronta')


In [ ]:
# @title 5) Composições mensais de NDVI (2024–2025)

def monthly_composite(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')
    comp = (s2_ndvi
            .filterDate(start, end)
            .median()
            .set('year', year)
            .set('month', month)
            .set('system:time_start', start.millis()))
    return comp

monthly_images = []
for y in range(ANO_INI, ANO_FIM + 1):
    for m in range(1, 13):
        monthly_images.append(monthly_composite(y, m))

monthly_ndvi = ee.ImageCollection(monthly_images).filterBounds(AOI)

print('✅ Composições mensais criadas:', monthly_ndvi.size().getInfo())


In [ ]:
# @title 6) NDVI máximo anual (2024 e 2025)

ndvi_max_2024 = (monthly_ndvi
                 .filter(ee.Filter.eq('year', 2024))
                 .max()
                 .clip(AOI)
                 .rename('NDVI_max_2024'))

ndvi_max_2025 = (monthly_ndvi
                 .filter(ee.Filter.eq('year', 2025))
                 .max()
                 .clip(AOI)
                 .rename('NDVI_max_2025'))

# NDVI máximo geral (opcional, para a máscara)
ndvi_max_all = (monthly_ndvi
                .max()
                .clip(AOI)
                .rename('NDVI_max'))

print('✅ NDVI máximo anual calculado')


In [ ]:
# @title 7) Máscara agrícola + filtros de ruído
# Threshold NDVI máximo
mask_ndvi = ndvi_max_all.gt(NDVI_MAX_THRESHOLD)

# Remoção de ruído por abertura/fechamento (filtro morfológico simples)
# Nota: ajuste do raio conforme necessário
radius = 1  # pixels
mask_clean = (mask_ndvi
              .focal_min(radius=radius)
              .focal_max(radius=radius))

print('✅ Máscara agrícola criada')


In [ ]:
# @title 8) Vetorização e filtragem por área
# Converte raster para vetores
vectors = mask_clean.selfMask().reduceToVectors(
    geometry=AOI,
    scale=S2_SCALE,
    geometryType='polygon',
    eightConnected=False,
    labelProperty='class',
    maxPixels=1e13
)

# Calcula área em hectares
vectors = vectors.map(lambda f: f.set({'area_ha': f.geometry().area().divide(1e4)}))

# Filtra por área mínima
vectors = vectors.filter(ee.Filter.gte('area_ha', AREA_MIN_HA))

# Filtra por área máxima (opcional)
if AREA_MAX_HA is not None:
    vectors = vectors.filter(ee.Filter.lte('area_ha', AREA_MAX_HA))

print('✅ Vetorização concluída')


In [ ]:
# @title 9) Visualização no mapa
Map = geemap.Map(center=[-15.2, -50.9], zoom=9)
Map.addLayer(ndvi_max_2024, {'min': 0, 'max': 1, 'palette': ['white','green']}, 'NDVI max 2024')
Map.addLayer(ndvi_max_2025, {'min': 0, 'max': 1, 'palette': ['white','green']}, 'NDVI max 2025')
Map.addLayer(mask_clean, {'palette': ['yellow']}, 'Mascara agrícola')
Map.addLayer(vectors, {'color': 'red'}, 'Talhões candidatos')
Map.addLayer(AOI, {'color': 'blue'}, 'AOI')
Map


In [ ]:
# @title 10) Exportar GeoJSON para Google Drive
# Exporta todos os talhões em um único arquivo (pode ser grande)

export_desc = f"{EXPORT_PREFIX}_{ANO_INI}_{ANO_FIM}"

export_task = ee.batch.Export.table.toDrive(
    collection=vectors,
    description=export_desc,
    folder=DRIVE_FOLDER,
    fileNamePrefix=export_desc,
    fileFormat='GeoJSON'
)

export_task.start()
print('✅ Exportação iniciada:', export_desc)


In [ ]:
# @title 11) (Opcional) Estratégia por tiles (grade)
# Use se a exportação total exceder limites.
# Define uma grade simples e exporta por tile.

# Parâmetros da grade (em graus) - ajuste conforme necessário
TILE_SIZE = 0.2

# Criar grade a partir da AOI
bounds = AOI.bounds().coordinates().getInfo()[0]
xs = [p[0] for p in bounds]
ys = [p[1] for p in bounds]
minx, maxx = min(xs), max(xs)
miny, maxy = min(ys), max(ys)

features = []
ix = 0
x = minx
while x < maxx:
    y = miny
    iy = 0
    while y < maxy:
        tile = ee.Geometry.Rectangle([x, y, x + TILE_SIZE, y + TILE_SIZE])
        inter = tile.intersection(AOI, 1)
        # Cria feature se houver interseção
        features.append(ee.Feature(inter, {'tile_id': f'{ix}_{iy}'}))
        y += TILE_SIZE
        iy += 1
    x += TILE_SIZE
    ix += 1

grid = ee.FeatureCollection(features)

print('Tiles criados:', grid.size().getInfo())

# Exportar por tile
# Observação: ajuste o número de tiles para evitar excesso de tarefas.

def export_tile(ft):
    tile_id = ft.get('tile_id').getInfo()
    tile_geom = ft.geometry()
    tile_vectors = vectors.filterBounds(tile_geom)
    desc = f"{EXPORT_PREFIX}_{tile_id}"
    task = ee.batch.Export.table.toDrive(
        collection=tile_vectors,
        description=desc,
        folder=DRIVE_FOLDER,
        fileNamePrefix=desc,
        fileFormat='GeoJSON'
    )
    task.start()
    print('✅ Exportação do tile iniciada:', tile_id)

# Exemplo: exportar apenas os primeiros 3 tiles
for ft in grid.toList(3).getInfo():
    export_tile(ee.Feature(ft))
